In [17]:
import os
import librosa
import numpy as np
import pandas as pd
import msaf
import msaf.algorithms
import pyloudnorm as pyln

In [18]:
def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=None)
    
    features = {}
    
    features["duration"] = librosa.get_duration(y=y, sr=sr)
    
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features["tempo"] = float(np.mean(tempo))
    
    features["rms_mean"] = np.mean(librosa.feature.rms(y=y))
    
    return features

In [19]:
def structural_features(filepath):
    boundaries, labels = msaf.process(filepath, boundaries_id="foote")
    
    segment_lengths = np.diff(boundaries)
    labels_array = np.array(labels)
    unique_labels = set(labels_array)
    
    ## Shannon entropy of segment labels to measure variability
    entropy = 0
    for l in unique_labels:
        if l != -1:
            p = list(labels_array).count(l) / len(labels_array)
            entropy += -p * np.log2(p)
    
    return {
        "num_segments": len(segment_lengths),
        "avg_segment_length": np.mean(segment_lengths),
        "segment_variability": entropy
    }

In [20]:
def loudness_features(filepath):
    y, sr = librosa.load(filepath, sr=None, mono=True)
    
    meter = pyln.Meter(sr)
    lufs = meter.integrated_loudness(y)
    
    rms = np.sqrt(np.mean(y**2))
    
    peak = np.max(np.abs(y))
    crest_db = 20 * np.log10(peak / rms) if rms > 0 else np.nan
    
    return {"lufs": lufs, "crest_db": crest_db}

In [21]:
song_path = "song_mp3/2010/Imagine Dragons - Radioactive (Lyric Video).mp3"

audio_feats = extract_features(song_path)
struct_feats = structural_features(song_path)
loud_feats = loudness_features(song_path)

combined = {**audio_feats, **struct_feats, **loud_feats}
combined["song"] = os.path.basename(song_path)
combined["decade"] = "2010s"

df_example = pd.DataFrame([combined])
df_example

AttributeError: module 'msaf' has no attribute 'algorithms'